In [1]:
# importações e configurações
import os
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

os.environ["BROWSER_PATH"] = r"C:\Program Files\Google\Chrome\Application\chrome.exe"

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")

TRAINING_RESULTS_DIR = ROOT / "results" / "model_training" / "randomized_search"
EVALUATION_RESULTS_DIR = ROOT / "results" / "model_evaluation"
ANALYSIS_DIR = ROOT / "results" / "results_analysis"
TABLES_DIR = ANALYSIS_DIR / "tables"
SVG_DIR = ANALYSIS_DIR / "figures" / "svg"
PNG_DIR = ANALYSIS_DIR / "figures" / "png"

for output_directory in [TABLES_DIR, SVG_DIR, PNG_DIR]:
    output_directory.mkdir(parents=True, exist_ok=True)

# limpeza dos artefatos anteriores
for previous_output in [
    *TABLES_DIR.glob("*.csv"),
    *SVG_DIR.glob("*.svg"),
    *PNG_DIR.glob("*.png"),
]:
    previous_output.unlink()

MODEL_ORDER = ["decision_tree", "random_forest", "xgboost", "linear_svm", "mlp"]
SCENARIO_ORDER = ["intra_dataset", "cross_dataset", "combined_to_unseen"]
TRAINING_SOURCE_ORDER = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
    "cicids2017_unsw_nb15",
    "cicids2017_iot23",
    "unsw_nb15_iot23",
]
TEST_SOURCE_ORDER = ["cicids2017", "unsw_nb15", "iot23"]

MODEL_LABELS = {
    "decision_tree": "Árvore de decisão",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "linear_svm": "SVM linear",
    "mlp": "MLP",
}
MODEL_ABBREVIATIONS = {
    "decision_tree": "DT",
    "random_forest": "RF",
    "xgboost": "XGB",
    "linear_svm": "SVM",
    "mlp": "MLP",
}
SCENARIO_LABELS = {
    "intra_dataset": "Intra-dataset",
    "cross_dataset": "Cross-dataset",
    "combined_to_unseen": "Combinado",
}
SCENARIO_FILE_LABELS = {
    "intra_dataset": "intra_dataset",
    "cross_dataset": "cross_dataset",
    "combined_to_unseen": "combined",
}
SOURCE_LABELS = {
    "cicids2017": "CICIDS2017",
    "unsw_nb15": "UNSW-NB15",
    "iot23": "IoT-23",
    "cicids2017_unsw_nb15": "CICIDS2017 + UNSW-NB15",
    "cicids2017_iot23": "CICIDS2017 + IoT-23",
    "unsw_nb15_iot23": "UNSW-NB15 + IoT-23",
}
SOURCE_ABBREVIATIONS = {
    "cicids2017": "CIC",
    "unsw_nb15": "UNSW",
    "iot23": "IoT",
    "cicids2017_unsw_nb15": "CIC+UNSW",
    "cicids2017_iot23": "CIC+IoT",
    "unsw_nb15_iot23": "UNSW+IoT",
}
MODEL_COLORS = {
    "decision_tree": "#4C78A8",
    "random_forest": "#F58518",
    "xgboost": "#54A24B",
    "linear_svm": "#E45756",
    "mlp": "#B279A2",
}

pd.set_option("display.max_columns", None)


def save_table(dataframe, filename):
    dataframe.to_csv(TABLES_DIR / filename, index=False, encoding="utf-8-sig")


def save_figure(
    figure,
    filename,
    width,
    height,
    top_margin=115,
    legend_y=1.02,
):
    figure.update_layout(
        template="plotly_white",
        font=dict(family="Arial", size=14),
        margin=dict(l=90, r=40, t=top_margin, b=75),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=legend_y,
            xanchor="center",
            x=0.5,
        ),
    )

    display(figure)
    figure.write_image(
        SVG_DIR / f"{filename}.svg", format="svg", width=width, height=height
    )
    figure.write_image(
        PNG_DIR / f"{filename}.png",
        format="png",
        width=width,
        height=height,
        scale=2,
    )


def format_mean_std(mean_value, std_value):
    return f"{mean_value:.4f} ± {std_value:.4f}"

In [2]:
# carregamento
training_summary = pd.concat(
    [
        pd.read_csv(TRAINING_RESULTS_DIR / model / "summary.csv")
        for model in MODEL_ORDER
    ],
    ignore_index=True,
)
evaluation = pd.read_csv(EVALUATION_RESULTS_DIR / "evaluation_summary.csv")

In [3]:
# tabelas de resultados
model_order_map = {model: index for index, model in enumerate(MODEL_ORDER)}
training_order_map = {source: index for index, source in enumerate(TRAINING_SOURCE_ORDER)}
test_order_map = {source: index for index, source in enumerate(TEST_SOURCE_ORDER)}

cv_source = training_summary.assign(
    model_order=lambda frame: frame["model"].map(model_order_map),
    training_order=lambda frame: frame["training_source"].map(training_order_map),
).sort_values(["training_order", "model_order"])

cross_validation_table = pd.DataFrame({
    "Conjunto de treinamento": cv_source["training_source"].map(SOURCE_LABELS),
    "Modelo": cv_source["model"].map(MODEL_ABBREVIATIONS),
    "F1-macro (média ± desvio-padrão)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            cv_source["f1_macro_mean"],
            cv_source["f1_macro_std"],
        )
    ],
    "Acurácia balanceada (média ± desvio-padrão)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            cv_source["balanced_accuracy_mean"],
            cv_source["balanced_accuracy_std"],
        )
    ],
})
save_table(cross_validation_table, "cross_validation_summary.csv")
display(cross_validation_table)

selected_hyperparameters = pd.DataFrame({
    "Conjunto de treinamento": cv_source["training_source"].map(SOURCE_LABELS),
    "Modelo": cv_source["model"].map(MODEL_ABBREVIATIONS),
    "Hiperparâmetros selecionados": cv_source["params"],
})
save_table(selected_hyperparameters, "selected_hyperparameters.csv")

evaluation_source = evaluation.assign(
    model_order=lambda frame: frame["model"].map(model_order_map),
    training_order=lambda frame: frame["training_source"].map(training_order_map),
    test_order=lambda frame: frame["test_source"].map(test_order_map),
).sort_values(["training_order", "test_order", "model_order"])

evaluation_table = pd.DataFrame({
    "Conjunto de treinamento": evaluation_source["training_source"].map(SOURCE_LABELS),
    "Conjunto de teste": evaluation_source["test_source"].map(SOURCE_LABELS),
    "Cenário de avaliação": evaluation_source["scenario_type"].map(SCENARIO_LABELS),
    "Modelo": evaluation_source["model"].map(MODEL_ABBREVIATIONS),
    "F1-macro": evaluation_source["f1_macro"].round(4),
    "Acurácia balanceada": evaluation_source["balanced_accuracy"].round(4),
    "Precisão (benigno)": evaluation_source["precision_benign"].round(4),
    "Recall (benigno)": evaluation_source["recall_benign"].round(4),
    "Precisão (ataque)": evaluation_source["precision_attack"].round(4),
    "Recall (ataque)": evaluation_source["recall_attack"].round(4),
})
save_table(evaluation_table, "evaluation_all.csv")

for scenario_type in SCENARIO_ORDER:
    scenario_table = evaluation_table[
        evaluation_table["Cenário de avaliação"].eq(SCENARIO_LABELS[scenario_type])
    ].drop(columns="Cenário de avaliação")
    save_table(
        scenario_table,
        f"evaluation_{SCENARIO_FILE_LABELS[scenario_type]}.csv",
    )
    print(f"Avaliação — {SCENARIO_LABELS[scenario_type]}")
    display(scenario_table)

,Conjunto de treinamento,Modelo,F1-macro (média ± desvio-padrão),Acurácia balanceada (média ± desvio-padrão)
0,CICIDS2017,DT,0.9909 ± 0.0005,0.9920 ± 0.0005
6,CICIDS2017,RF,0.9929 ± 0.0002,0.9959 ± 0.0001
12,CICIDS2017,XGB,0.9924 ± 0.0002,0.9946 ± 0.0004
18,CICIDS2017,SVM,0.8909 ± 0.0020,0.9357 ± 0.0013
24,CICIDS2017,MLP,0.9911 ± 0.0004,0.9938 ± 0.0006
1,UNSW-NB15,DT,0.9787 ± 0.0011,0.9890 ± 0.0006
7,UNSW-NB15,RF,0.9800 ± 0.0010,0.9911 ± 0.0005
13,UNSW-NB15,XGB,0.9800 ± 0.0011,0.9912 ± 0.0005
19,UNSW-NB15,SVM,0.9359 ± 0.0013,0.9229 ± 0.0012
25,UNSW-NB15,MLP,0.9786 ± 0.0013,0.9897 ± 0.0008


Avaliação — Intra-dataset


,Conjunto de treinamento,Conjunto de teste,Modelo,F1-macro,Acurácia balanceada,Precisão (benigno),Recall (benigno),Precisão (ataque),Recall (ataque)
0,CICIDS2017,CICIDS2017,DT,0.9911,0.9922,0.9972,0.9957,0.9828,0.9888
3,CICIDS2017,CICIDS2017,RF,0.9933,0.9964,0.9994,0.9952,0.9812,0.9975
4,CICIDS2017,CICIDS2017,XGB,0.9923,0.9944,0.9983,0.9955,0.9821,0.9934
1,CICIDS2017,CICIDS2017,SVM,0.8900,0.9350,0.9883,0.9133,0.7339,0.9568
2,CICIDS2017,CICIDS2017,MLP,0.9916,0.9942,0.9984,0.9948,0.9797,0.9935
50,UNSW-NB15,UNSW-NB15,DT,0.9798,0.9899,0.9987,0.9847,0.9422,0.9950
53,UNSW-NB15,UNSW-NB15,RF,0.9807,0.9915,0.9996,0.9846,0.9418,0.9985
54,UNSW-NB15,UNSW-NB15,XGB,0.9807,0.9916,0.9997,0.9846,0.9417,0.9987
51,UNSW-NB15,UNSW-NB15,SVM,0.9360,0.9226,0.9656,0.9855,0.9366,0.8597
52,UNSW-NB15,UNSW-NB15,MLP,0.9796,0.9907,0.9993,0.9840,0.9396,0.9974


Avaliação — Cross-dataset


,Conjunto de treinamento,Conjunto de teste,Modelo,F1-macro,Acurácia balanceada,Precisão (benigno),Recall (benigno),Precisão (ataque),Recall (ataque)
10,CICIDS2017,UNSW-NB15,DT,0.4427,0.4916,0.7973,0.9799,0.0402,0.0034
13,CICIDS2017,UNSW-NB15,RF,0.4444,0.5000,0.8000,0.9999,0.0000,0.0000
14,CICIDS2017,UNSW-NB15,XGB,0.4444,0.5000,0.8000,0.9999,0.0000,0.0000
11,CICIDS2017,UNSW-NB15,SVM,0.4413,0.4906,0.7969,0.9790,0.0253,0.0022
12,CICIDS2017,UNSW-NB15,MLP,0.4445,0.4997,0.7999,0.9992,0.0714,0.0002
5,CICIDS2017,IoT-23,DT,0.0867,0.0636,0.3162,0.1141,0.0037,0.0131
8,CICIDS2017,IoT-23,RF,0.0882,0.0627,0.3229,0.1184,0.0020,0.0069
9,CICIDS2017,IoT-23,XGB,0.0866,0.0615,0.3187,0.1161,0.0020,0.0070
6,CICIDS2017,IoT-23,SVM,0.1239,0.1516,0.3507,0.1088,0.0517,0.1945
7,CICIDS2017,IoT-23,MLP,0.4451,0.4939,0.7980,0.9827,0.0686,0.0051


Avaliação — Combinado


,Conjunto de treinamento,Conjunto de teste,Modelo,F1-macro,Acurácia balanceada,Precisão (benigno),Recall (benigno),Precisão (ataque),Recall (ataque)
20,CICIDS2017 + UNSW-NB15,IoT-23,DT,0.0899,0.0666,0.3241,0.1180,0.0043,0.0152
23,CICIDS2017 + UNSW-NB15,IoT-23,RF,0.0205,0.0167,0.0869,0.0236,0.0025,0.0098
24,CICIDS2017 + UNSW-NB15,IoT-23,XGB,0.0173,0.0142,0.0738,0.0198,0.0022,0.0087
21,CICIDS2017 + UNSW-NB15,IoT-23,SVM,0.1245,0.1530,0.3515,0.1088,0.0524,0.1972
22,CICIDS2017 + UNSW-NB15,IoT-23,MLP,0.1510,0.4106,0.1902,0.0112,0.1700,0.8100
15,CICIDS2017 + IoT-23,UNSW-NB15,DT,0.4828,0.5168,0.8055,0.9928,0.5856,0.0408
18,CICIDS2017 + IoT-23,UNSW-NB15,RF,0.4733,0.5137,0.8044,0.9987,0.8467,0.0287
19,CICIDS2017 + IoT-23,UNSW-NB15,XGB,0.4449,0.5002,0.8001,0.9998,0.4167,0.0005
16,CICIDS2017 + IoT-23,UNSW-NB15,SVM,0.4449,0.4973,0.7991,0.9923,0.0707,0.0023
17,CICIDS2017 + IoT-23,UNSW-NB15,MLP,0.4901,0.5211,0.8068,0.9943,0.6774,0.0479


In [4]:
# gráfico geral
EVALUATION_ROUTES_BY_SCENARIO = {
    "intra_dataset": ["CIC → CIC", "UNSW → UNSW", "IoT → IoT"],
    "cross_dataset": [
        "CIC → UNSW",
        "CIC → IoT",
        "UNSW → CIC",
        "UNSW → IoT",
        "IoT → CIC",
        "IoT → UNSW",
    ],
    "combined_to_unseen": [
        "CIC+UNSW → IoT",
        "CIC+IoT → UNSW",
        "UNSW+IoT → CIC",
    ],
}

general_plot_data = evaluation.copy()
general_plot_data["evaluation_route"] = (
    general_plot_data["training_source"].map(SOURCE_ABBREVIATIONS)
    + " → "
    + general_plot_data["test_source"].map(SOURCE_ABBREVIATIONS)
)

general_figure = make_subplots(
    rows=3,
    cols=1,
    vertical_spacing=0.11,
    subplot_titles=[SCENARIO_LABELS[scenario] for scenario in SCENARIO_ORDER],
)
for row_index, scenario_type in enumerate(SCENARIO_ORDER, start=1):
    routes = EVALUATION_ROUTES_BY_SCENARIO[scenario_type]
    scenario_data = general_plot_data[
        general_plot_data["scenario_type"].eq(scenario_type)
    ]

    for model, offset in zip(MODEL_ORDER, [-0.24, -0.12, 0, 0.12, 0.24]):
        model_data = scenario_data[scenario_data["model"].eq(model)].set_index(
            "evaluation_route"
        ).loc[routes]
        general_figure.add_trace(
            go.Scatter(
                x=[index + offset for index in range(len(routes))],
                y=model_data["f1_macro"],
                mode="markers",
                name=MODEL_LABELS[model],
                showlegend=row_index == 1,
                marker=dict(size=11, color=MODEL_COLORS[model]),
            ),
            row=row_index,
            col=1,
        )

    general_figure.update_xaxes(
        tickmode="array",
        tickvals=list(range(len(routes))),
        ticktext=routes,
        range=[-0.55, len(routes) - 0.45],
        row=row_index,
        col=1,
    )
    general_figure.update_yaxes(
        title_text="F1-macro",
        range=[0, 1.02],
        dtick=0.2,
        row=row_index,
        col=1,
    )
save_figure(
    general_figure,
    "f1_macro_all_training_test_combinations",
    width=1100,
    height=1150,
)

In [5]:
# comparação entre treinamentos individuais e combinado
TARGET_TRAINING_SOURCES = {
    "cicids2017": ("unsw_nb15", "iot23", "unsw_nb15_iot23"),
    "unsw_nb15": ("cicids2017", "iot23", "cicids2017_iot23"),
    "iot23": ("cicids2017", "unsw_nb15", "cicids2017_unsw_nb15"),
}

comparison_rows = []
plot_rows = []
for test_source, sources in TARGET_TRAINING_SOURCES.items():
    individual_a, individual_b, combined = sources
    for model in MODEL_ORDER:
        rows = evaluation[
            evaluation["test_source"].eq(test_source)
            & evaluation["model"].eq(model)
            & evaluation["training_source"].isin(sources)
        ].set_index("training_source")
        f1_a = rows.loc[individual_a, "f1_macro"]
        f1_b = rows.loc[individual_b, "f1_macro"]
        f1_combined = rows.loc[combined, "f1_macro"]
        balanced_a = rows.loc[individual_a, "balanced_accuracy"]
        balanced_b = rows.loc[individual_b, "balanced_accuracy"]
        balanced_combined = rows.loc[combined, "balanced_accuracy"]

        comparison_rows.append({
            "test_source": test_source,
            "model": model,
            "individual_a": individual_a,
            "f1_a": f1_a,
            "balanced_a": balanced_a,
            "individual_b": individual_b,
            "f1_b": f1_b,
            "balanced_b": balanced_b,
            "combined": combined,
            "f1_combined": f1_combined,
            "balanced_combined": balanced_combined,
            "f1_delta": f1_combined - max(f1_a, f1_b),
            "balanced_delta": balanced_combined - max(balanced_a, balanced_b),
        })

        for configuration, training_source in [
            ("A (individual)", individual_a),
            ("B (individual)", individual_b),
            ("A + B (combinado)", combined),
        ]:
            plot_rows.append({
                "test_source": test_source,
                "model": model,
                "configuration": configuration,
                "training_source": training_source,
                "f1_macro": rows.loc[training_source, "f1_macro"],
            })

combined_comparison = pd.DataFrame(comparison_rows)
combined_comparison_table = pd.DataFrame({
    "Conjunto de teste": combined_comparison["test_source"].map(SOURCE_LABELS),
    "Modelo": combined_comparison["model"].map(MODEL_ABBREVIATIONS),
    "Origem individual A": combined_comparison["individual_a"].map(SOURCE_LABELS),
    "F1-macro — A": combined_comparison["f1_a"].round(4),
    "Acurácia balanceada — A": combined_comparison["balanced_a"].round(4),
    "Origem individual B": combined_comparison["individual_b"].map(SOURCE_LABELS),
    "F1-macro — B": combined_comparison["f1_b"].round(4),
    "Acurácia balanceada — B": combined_comparison["balanced_b"].round(4),
    "Origens combinadas A + B": combined_comparison["combined"].map(SOURCE_LABELS),
    "F1-macro — A + B": combined_comparison["f1_combined"].round(4),
    "Acurácia balanceada — A + B": combined_comparison["balanced_combined"].round(4),
    "Δ F1-macro": combined_comparison["f1_delta"].round(4),
    "Δ acurácia balanceada": combined_comparison["balanced_delta"].round(4),
})
save_table(
    combined_comparison_table,
    "combined_vs_individual_training_sources.csv",
)
display(combined_comparison_table)

comparison_plot_data = pd.DataFrame(plot_rows)
configuration_order = [
    "A (individual)",
    "B (individual)",
    "A + B (combinado)",
]
configuration_colors = {
    "A (individual)": "#4C78A8",
    "B (individual)": "#F58518",
    "A + B (combinado)": "#54A24B",
}
panel_titles = []
for test_source in TEST_SOURCE_ORDER:
    individual_a, individual_b, combined = TARGET_TRAINING_SOURCES[test_source]
    panel_titles.append(
        f"Teste: {SOURCE_LABELS[test_source]}<br>"
        f"A: {SOURCE_LABELS[individual_a]} · "
        f"B: {SOURCE_LABELS[individual_b]} · "
        f"A+B: {SOURCE_LABELS[combined]}"
    )

comparison_figure = make_subplots(
    rows=3,
    cols=1,
    vertical_spacing=0.13,
    subplot_titles=panel_titles,
)
for row_index, test_source in enumerate(TEST_SOURCE_ORDER, start=1):
    test_data = comparison_plot_data[
        comparison_plot_data["test_source"].eq(test_source)
    ]
    for configuration in configuration_order:
        configuration_data = test_data[
            test_data["configuration"].eq(configuration)
        ].set_index("model").loc[MODEL_ORDER]
        comparison_figure.add_trace(
            go.Bar(
                x=[MODEL_LABELS[model] for model in MODEL_ORDER],
                y=configuration_data["f1_macro"],
                name=configuration,
                showlegend=row_index == 1,
                marker_color=configuration_colors[configuration],
            ),
            row=row_index,
            col=1,
        )

    comparison_figure.update_yaxes(
        title_text="F1-macro",
        range=[0, 1.02],
        dtick=0.2,
        row=row_index,
        col=1,
    )

comparison_figure.update_layout(barmode="group", bargap=0.18, bargroupgap=0.05)
save_figure(
    comparison_figure,
    "individual_and_combined_f1_macro",
    width=1100,
    height=1250,
    top_margin=180,
    legend_y=1.10,
)

,Conjunto de teste,Modelo,Origem individual A,F1-macro — A,Acurácia balanceada — A,Origem individual B,F1-macro — B,Acurácia balanceada — B,Origens combinadas A + B,F1-macro — A + B,Acurácia balanceada — A + B,Δ F1-macro,Δ acurácia balanceada
0,CICIDS2017,DT,UNSW-NB15,0.4297,0.4249,IoT-23,0.7673,0.7483,UNSW-NB15 + IoT-23,0.4208,0.4494,-0.3465,-0.2989
1,CICIDS2017,RF,UNSW-NB15,0.4241,0.4201,IoT-23,0.4517,0.4983,UNSW-NB15 + IoT-23,0.4877,0.4929,0.0360,-0.0054
2,CICIDS2017,XGB,UNSW-NB15,0.4029,0.3970,IoT-23,0.4511,0.4923,UNSW-NB15 + IoT-23,0.5043,0.5048,0.0532,0.0125
3,CICIDS2017,SVM,UNSW-NB15,0.6994,0.7015,IoT-23,0.4850,0.4853,UNSW-NB15 + IoT-23,0.4954,0.6804,-0.2040,-0.0211
4,CICIDS2017,MLP,UNSW-NB15,0.6319,0.6659,IoT-23,0.4602,0.4964,UNSW-NB15 + IoT-23,0.4410,0.4466,-0.1909,-0.2193
5,UNSW-NB15,DT,CICIDS2017,0.4427,0.4916,IoT-23,0.5549,0.5538,CICIDS2017 + IoT-23,0.4828,0.5168,-0.0720,-0.0370
6,UNSW-NB15,RF,CICIDS2017,0.4444,0.5000,IoT-23,0.5061,0.5279,CICIDS2017 + IoT-23,0.4733,0.5137,-0.0328,-0.0142
7,UNSW-NB15,XGB,CICIDS2017,0.4444,0.5000,IoT-23,0.3424,0.3233,CICIDS2017 + IoT-23,0.4449,0.5002,0.0005,0.0002
8,UNSW-NB15,SVM,CICIDS2017,0.4413,0.4906,IoT-23,0.4859,0.5079,CICIDS2017 + IoT-23,0.4449,0.4973,-0.0410,-0.0106
9,UNSW-NB15,MLP,CICIDS2017,0.4445,0.4997,IoT-23,0.4543,0.4567,CICIDS2017 + IoT-23,0.4901,0.5211,0.0359,0.0214
